Downloading the data from Yahoo Finance

In [ ]:
import yfinance as yf
import pandas as pd
import os
from datetime import date

def get_latest_date():
    today = date.today()
    return today.strftime("%Y-%m-%d")


def clean_and_save_data(data, filepath):
    # Reset index to include the 'Date' column in the DataFrame
    data.reset_index(inplace=True)

    # Remove column-level names, if any
    data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
    
    # Remove rows with any non-numeric values in the data columns (excluding 'Date')
    for col in data.columns[1:]:  # Skip the 'Date' column
        data[col] = pd.to_numeric(data[col], errors='coerce')
    data = data.dropna()  # Drop rows with NaN values

    # Save the cleaned data to CSV
    data.to_csv(filepath, index=False)
    print(f"Cleaned data saved to: {filepath}")


# Function to download and save stock data
def download_stock_data(interval, folder):
    base_path = "C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS"
    filepath = os.path.join(base_path, "indicesstocks.csv")
    start_date = "2020-01-01"
    end_date = get_latest_date()
    with open(filepath) as f:
        for line in f:
            if "," not in line:
                continue
            symbols = line.split(",")
            for symbol in symbols:
                symbol = symbol.strip()  # Remove any whitespace
                try:
                    # Download stock data
                    data = yf.download(symbol, start=start_date, end=end_date, interval=interval)
                    ticketfilename = symbol.replace(".", "_")
                    save_path = os.path.join(base_path, folder, f"{ticketfilename}.csv")
                    # Clean the data and save
                    clean_and_save_data(data, save_path)
                except Exception as e:
                    print(f"Error downloading data for {symbol}: {e}")

# Main function
if __name__ == '__main__':
    print("Starting to download and clean stock data...")
    # Daily data
    download_stock_data(interval='1d', folder='Daily_data')
    # Weekly data
    download_stock_data(interval='1wk', folder='Weekly_data')
    # Monthly data
    download_stock_data(interval='1mo', folder='Monthly_data')
    print("All stock data downloaded and cleaned successfully!")


*****GENERATING INDICES DATA*****

In [7]:
import pandas as pd
import datetime as dt
from pathlib import Path
import pandas_ta as ta
from tabulate import tabulate

def append_row(df, row):
    """Appends a new row to a pandas DataFrame."""
    # return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)
    df = pd.concat([df, pd.DataFrame([row], columns=row.index)], ignore_index=True)
    return df

def getRSI14(csvfilename):
    """Calculates the RSI (14 period) for a given CSV file."""
    if Path(csvfilename).is_file():
        try:
            df = pd.read_csv(csvfilename)
            if df.empty:
                return 0.00, 0.00
            else:
                df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
                df = df.dropna(subset=['Close'])  # removing rows where 'Close' is NaN
                if df.empty:
                    return 0.00, 0.00
                df['rsi14'] = ta.rsi(df['Close'], length=14)
                if pd.isna(df['rsi14'].iloc[-1]):
                    return 0.00, 0.00
                else:
                    rsival = df['rsi14'].iloc[-1].round(2)
                    ltp = df['Close'].iloc[-1].round(2)
                return rsival, ltp
        except Exception as e:
            print(f"Error reading {csvfilename}: {e}")
            return 0.00, 0.00
    else:
        print(f"File does not exist: {csvfilename}")
        return 0.00, 0.00

def dayweekmonth_datasets(symbol, symbolname):
    """Calculates RSI for daily, weekly, and monthly data."""
    if symbol.endswith('.NS'):
        symbol = symbol.replace(".NS", "_NS")

    base_path = Path("C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS")
    daylocationstr = base_path / "Daily_data" / f"{symbol}.csv"
    weeklocationstr = base_path / "Weekly_data" / f"{symbol}.csv"
    monthlocationstr = base_path / "Monthly_data" / f"{symbol}.csv"

    cday = dt.datetime.today().strftime('%d/%m/%Y')
    dayrsi14, dltp = getRSI14(daylocationstr)
    weekrsi14, wltp = getRSI14(weeklocationstr)
    monthrsi14, mltp = getRSI14(monthlocationstr)

    new_row = pd.Series({
        'entrydate': cday,
        'indexcode': symbol,
        'indexname': symbolname.strip(),
        'dayrsi14': dayrsi14,
        'weekrsi14': weekrsi14,
        'monthrsi14': monthrsi14
    })
    return new_row

def generateGFS(scripttype):
    """Generates the GFS report based on the provided scripttype."""
    indicesdf = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'dayrsi14', 'weekrsi14', 'monthrsi14'])

    base_path = Path("C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS")
    fname = base_path / scripttype
    csvfilename = base_path / f"GFS_{scripttype}.csv"

    try:
        with open(fname) as f:
            for line in f:
                if "," not in line:
                    continue
                parts = line.split(",")
                if len(parts) < 2:
                    continue  # Skip lines that don't have enough parts
                symbol, symbolname = parts[0].strip(), parts[1].strip()
                symbol = symbol.strip()
                new_row = dayweekmonth_datasets(symbol, symbolname)
                indicesdf = append_row(indicesdf, new_row)
    except Exception as e:
        print(f"Error processing {fname}: {e}")

    indicesdf.to_csv(csvfilename, index=False)
    return indicesdf

# Generate the GFS report
df3 = generateGFS("indicesdf.csv")

# Display the DataFrame in tabular format using tabulate
print(tabulate(df3, headers='keys', tablefmt='fancy_grid'))


C:\Users\manoj\AppData\Local\Temp\ipykernel_20992\435983743.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row], columns=row.index)], ignore_index=True)


╒════╤═════════════╤═════════════╤══════════════════════╤════════════╤═════════════╤══════════════╕
│    │ entrydate   │ indexcode   │ indexname            │   dayrsi14 │   weekrsi14 │   monthrsi14 │
╞════╪═════════════╪═════════════╪══════════════════════╪════════════╪═════════════╪══════════════╡
│  0 │ 14/02/2025  │ ^NSEI       │ NIFTY 50             │      40.63 │       41.24 │        59.26 │
├────┼─────────────┼─────────────┼──────────────────────┼────────────┼─────────────┼──────────────┤
│  1 │ 14/02/2025  │ ^CNXAUTO    │ NIFTY AUTO           │      41.39 │       42.8  │        62.65 │
├────┼─────────────┼─────────────┼──────────────────────┼────────────┼─────────────┼──────────────┤
│  2 │ 14/02/2025  │ ^NSEBANK    │ NIFTY BANK           │      46.29 │       45.03 │        57.33 │
├────┼─────────────┼─────────────┼──────────────────────┼────────────┼─────────────┼──────────────┤
│  3 │ 14/02/2025  │ ^CNXFMCG    │ NIFTY FMCG           │      30.47 │       34.87 │        75.81 │


######### GFS LOGIC FOR INDICES - TESTING #########

In [8]:
import pandas as pd
import datetime as dt
from pathlib import Path
import pandas_ta as ta
from tabulate import tabulate
import os

def append_row(df, row):
    """Appends a new row to a pandas DataFrame.

    Args:
        df (pd.DataFrame): The existing DataFrame.
        row (pd.Series): The new row to append.

    Returns:
        pd.DataFrame: The DataFrame with the new row appended.
    """
    return pd.concat([
        df,
        pd.DataFrame([row], columns=row.index)
    ]).reset_index(drop=True)

def getRSI14_and_BB(csvfilename):
    """Calculates RSI (14 period) and Bollinger Bands (20 period, 2 std.dev) for a given CSV file.

    Args:
        csvfilename (str): Path to the CSV file.

    Returns:
        tuple: A tuple containing RSI value, last close price, lower band, and middle band.
        (float, float, float, float)
    """
    if Path(csvfilename).is_file():
        try:
            df = pd.read_csv(csvfilename)
            if df.empty or 'Close' not in df.columns:
                return 0.00, 0.00, 0.00, 0.00
            else:
                df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
                df['rsi14'] = ta.rsi(df['Close'], length=14)
                bb = ta.bbands(df['Close'], length=20)
                if bb is None or df['rsi14'] is None:
                    return 0.00, 0.00, 0.00, 0.00
                df['lowerband'] = bb['BBL_20_2.0']
                df['middleband'] = bb['BBM_20_2.0']
                if pd.isna(df['rsi14'].iloc[-1]) or pd.isna(df['lowerband'].iloc[-1]) or pd.isna(df['middleband'].iloc[-1]):
                    return 0.00, 0.00, 0.00, 0.00
                else:
                    rsival = df['rsi14'].iloc[-1].round(2)
                    ltp = df['Close'].iloc[-1].round(2)
                    lowerband = df['lowerband'].iloc[-1].round(2)
                    middleband = df['middleband'].iloc[-1].round(2)
                    return rsival, ltp, lowerband, middleband
        except Exception as e:
            print(f"Error reading {csvfilename}: {e}")
            return 0.00, 0.00, 0.00, 0.00
    else:
        print(f"File does not exist: {csvfilename}")
        return 0.00, 0.00, 0.00, 0.00

def dayweekmonth_datasets(symbol, symbolname):
    """Calculates RSI, Bollinger Bands, and other metrics for daily, weekly, and monthly data.

    Args:
        symbol (str): The symbol of the asset.
        symbolname (str): The name of the asset.

    Returns:
        pd.Series: A Series containing the calculated metrics.
    """
    daylocationstr = f'DATASETS/Daily_data/{symbol}.csv'
    weeklocationstr = f'DATASETS/Weekly_data/{symbol}.csv'
    monthlocationstr = f'DATASETS/Monthly_data/{symbol}.csv'

    cday = dt.datetime.today().strftime('%d/%m/%Y')
    dayrsi14, dltp, daylowerband, daymiddleband = getRSI14_and_BB(daylocationstr)
    weekrsi14, wltp, weeklowerband, weekmiddleband = getRSI14_and_BB(weeklocationstr)
    monthrsi14, mltp, monthlowerband, monthmiddleband = getRSI14_and_BB(monthlocationstr)

    new_row = pd.Series({
        'entrydate': cday,
        'indexcode': symbol,
        'indexname': symbolname,
        'dayrsi14': dayrsi14,
        'weekrsi14': weekrsi14,
        'monthrsi14': monthrsi14,
        'dltp': dltp,
        'daylowerband': daylowerband,
        'daymiddleband': daymiddleband,
        'weeklowerband': weeklowerband,
        'weekmiddleband': weekmiddleband,
        'monthlowerband': monthlowerband,
        'monthmiddleband': monthmiddleband
    })
    return new_row

def generateGFS(scripttype):
    """Generates the GFS report based on the provided scripttype.

    Args:
        scripttype (str): The name of the scripttype file.

    Returns:
        pd.DataFrame: The DataFrame containing the GFS report.
    """
    indicesdf = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

    fname = f'DATASETS/{scripttype}.csv'
    csvfilename = f'GFS_{scripttype}.csv'
    try:
        with open(fname) as f:
            for line in f:
                if "," not in line:
                    continue
                symbol, symbolname = line.split(",")[0], line.split(",")[1]
                symbol = symbol.replace("\n", "")
                new_row = dayweekmonth_datasets(symbol, symbolname)
                indicesdf = append_row(indicesdf, new_row)
    except Exception as e:
        print(f"Error processing {fname}: {e}")

    indicesdf.to_csv(csvfilename, index=False)
    return indicesdf

# Assuming 'DATASETS/indicesdf.csv' exists and is in the correct format
df3 = generateGFS('indicesdf')

df4 = df3.loc[
    df3['monthrsi14'].between(40, 60) &
    df3['weekrsi14'].between(40, 60) &
    df3['dayrsi14'].between(40, 60)
]

df4 = df4.sort_values(by=['dayrsi14'], ascending=True)


if df4.empty:
    print("\033[1mNO INDICES QUALIFIES THE GFS CRITERIA\033[0m")
else:
    # Save the filtered results to a file
    filtered_filename = 'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\filtered_indices.csv'
    df4.to_csv(filtered_filename, index=False)

    
    # Print the table
    print(tabulate(df4, headers='keys', tablefmt='fancy_grid', showindex=False))
    print(f"\033[1mFiltered indices have been saved to {filtered_filename}\033[0m")


C:\Users\manoj\AppData\Local\Temp\ipykernel_20992\1245320971.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([


╒═════════════╤═════════════╤═════════════╤════════════╤═════════════╤══════════════╤══════════╤════════════════╤═════════════════╤═════════════════╤══════════════════╤══════════════════╤═══════════════════╕
│ entrydate   │ indexcode   │ indexname   │   dayrsi14 │   weekrsi14 │   monthrsi14 │     dltp │   daylowerband │   daymiddleband │   weeklowerband │   weekmiddleband │   monthlowerband │   monthmiddleband │
╞═════════════╪═════════════╪═════════════╪════════════╪═════════════╪══════════════╪══════════╪════════════════╪═════════════════╪═════════════════╪══════════════════╪══════════════════╪═══════════════════╡
│ 14/02/2025  │ ^NSEI       │ NIFTY 50    │      40.63 │       41.24 │        59.26 │ 23031.4  │       22762.5  │        23261.1  │        22762.2  │         23985.8  │         18465.5  │          22464.5  │
├─────────────┼─────────────┼─────────────┼────────────┼─────────────┼──────────────┼──────────┼────────────────┼─────────────────┼─────────────────┼──────────────────┼

*****GFS LOGIC FOR INDICES - CORRECT CODE IS DOWN BELOW ***** 

In [ ]:
# import pandas as pd
# import datetime as dt
# from pathlib import Path
# import pandas_ta as ta
# from tabulate import tabulate
# import os

# def append_row(df, row):
#     """Appends a new row to a pandas DataFrame.

#     Args:
#         df (pd.DataFrame): The existing DataFrame.
#         row (pd.Series): The new row to append.

#     Returns:
#         pd.DataFrame: The DataFrame with the new row appended.
#     """
#     return pd.concat([
#         df,
#         pd.DataFrame([row], columns=row.index)
#     ]).reset_index(drop=True)

# def getRSI14_and_BB(csvfilename):
#     """Calculates RSI (14 period) and Bollinger Bands (20 period, 2 std.dev) for a given CSV file.

#     Args:
#         csvfilename (str): Path to the CSV file.

#     Returns:
#         tuple: A tuple containing RSI value, last close price, lower band, and middle band.
#         (float, float, float, float)
#     """
#     if Path(csvfilename).is_file():
#         try:
#             df = pd.read_csv(csvfilename)
#             if df.empty or 'Close' not in df.columns:
#                 return 0.00, 0.00, 0.00, 0.00
#             else:
#                 df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
#                 df['rsi14'] = ta.rsi(df['Close'], length=14)
#                 bb = ta.bbands(df['Close'], length=20)
#                 if bb is None or df['rsi14'] is None:
#                     return 0.00, 0.00, 0.00, 0.00
#                 df['lowerband'] = bb['BBL_20_2.0']
#                 df['middleband'] = bb['BBM_20_2.0']
#                 if pd.isna(df['rsi14'].iloc[-1]) or pd.isna(df['lowerband'].iloc[-1]) or pd.isna(df['middleband'].iloc[-1]):
#                     return 0.00, 0.00, 0.00, 0.00
#                 else:
#                     rsival = df['rsi14'].iloc[-1].round(2)
#                     ltp = df['Close'].iloc[-1].round(2)
#                     lowerband = df['lowerband'].iloc[-1].round(2)
#                     middleband = df['middleband'].iloc[-1].round(2)
#                     return rsival, ltp, lowerband, middleband
#         except Exception as e:
#             print(f"Error reading {csvfilename}: {e}")
#             return 0.00, 0.00, 0.00, 0.00
#     else:
#         print(f"File does not exist: {csvfilename}")
#         return 0.00, 0.00, 0.00, 0.00

# def dayweekmonth_datasets(symbol, symbolname):
#     """Calculates RSI, Bollinger Bands, and other metrics for daily, weekly, and monthly data.

#     Args:
#         symbol (str): The symbol of the asset.
#         symbolname (str): The name of the asset.

#     Returns:
#         pd.Series: A Series containing the calculated metrics.
#     """
#     daylocationstr = f'DATASETS/Daily_data/{symbol}.csv'
#     weeklocationstr = f'DATASETS/Weekly_data/{symbol}.csv'
#     monthlocationstr = f'DATASETS/Monthly_data/{symbol}.csv'

#     cday = dt.datetime.today().strftime('%d/%m/%Y')
#     dayrsi14, dltp, daylowerband, daymiddleband = getRSI14_and_BB(daylocationstr)
#     weekrsi14, wltp, weeklowerband, weekmiddleband = getRSI14_and_BB(weeklocationstr)
#     monthrsi14, mltp, monthlowerband, monthmiddleband = getRSI14_and_BB(monthlocationstr)

#     new_row = pd.Series({
#         'entrydate': cday,
#         'indexcode': symbol,
#         'indexname': symbolname,
#         'dayrsi14': dayrsi14,
#         'weekrsi14': weekrsi14,
#         'monthrsi14': monthrsi14,
#         'dltp': dltp,
#         'daylowerband': daylowerband,
#         'daymiddleband': daymiddleband,
#         'weeklowerband': weeklowerband,
#         'weekmiddleband': weekmiddleband,
#         'monthlowerband': monthlowerband,
#         'monthmiddleband': monthmiddleband
#     })
#     return new_row

# def generateGFS(scripttype):
#     """Generates the GFS report based on the provided scripttype.

#     Args:
#         scripttype (str): The name of the scripttype file.

#     Returns:
#         pd.DataFrame: The DataFrame containing the GFS report.
#     """
#     indicesdf = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

#     fname = f'DATASETS/{scripttype}.csv'
#     csvfilename = f'GFS_{scripttype}.csv'
#     try:
#         with open(fname) as f:
#             for line in f:
#                 if "," not in line:
#                     continue
#                 symbol, symbolname = line.split(",")[0], line.split(",")[1]
#                 symbol = symbol.replace("\n", "")
#                 new_row = dayweekmonth_datasets(symbol, symbolname)
#                 indicesdf = append_row(indicesdf, new_row)
#     except Exception as e:
#         print(f"Error processing {fname}: {e}")

#     indicesdf.to_csv(csvfilename, index=False)
#     return indicesdf

# # Assuming 'DATASETS/indicesdf.csv' exists and is in the correct format
# df3 = generateGFS('indicesdf')

# df4 = df3.loc[
#     (df3['monthrsi14'] >= 60.00) &
#     (df3['weekrsi14'] >= 60.00) &
#     df3['dayrsi14'].between(40, 50) &
#     (df3['dltp'] > df3['daylowerband']) &
#     (df3['dltp'] < df3['daymiddleband'])
# ]

# df4 = df4.sort_values(by=['dayrsi14'], ascending=True)

# if df4.empty:
#     print("\033[1mNO INDICES QUALIFIES THE GFS CRITERIA\033[0m")
# else:
#     print(tabulate(df4, headers='keys', tablefmt='fancy_grid', showindex=False))

*****STOCKS DATA IN INDICES*****

In [9]:
import os
from tabulate import tabulate

def traverse_files(file1_path, file2_path):
    # Read the first file
    with open(file1_path, 'r') as f1:
        file1_data = f1.readlines()

    # Read the second file
    with open(file2_path, 'r') as f2:
        file2_data = f2.readlines()

    # Use a dictionary to store the second file's data by its first column values
    file2_dict = {}
    for line in file2_data:
        # Strip whitespace and split by commas
        parts = line.strip().split(',')
        if parts:  # Ensure line is not empty
            # Store the rest of the data by the first column
            file2_dict[parts[0]] = parts[1:]

    # Store matched results
    matched_results = []

    # Traverse through the first file and check for matches in the second file
    for line in file1_data:
        # Strip whitespace and split by commas
        parts = line.strip().split(',')
        if parts and parts[0] in file2_dict:
            # Prepare the matched result
            matched_results.append([parts[0]] + file2_dict[parts[0]])

    # Print the matched results in a tabular format
    if matched_results:
        headers = ['Index Code'] + [f'Column {i+1}' for i in range(len(matched_results[0]) - 1)]
        print(tabulate(matched_results, headers=headers,tablefmt='fancy_grid')) #tablefmt='psql'))
    else:
        print("No matches found!")

# Specify the paths to your files
file1_path = r"C:\\Users\manoj\Downloads\Major project data\Major pro source codes\DATASETS\indicesdf.csv"  # Replace with your actual file path
file2_path = r"C:\\Users\manoj\Downloads\Major project data\Major pro source codes\DATASETS\indicesstocks.csv"  # Replace with your actual file path

# Call the function
traverse_files(file1_path, file2_path)

╒══════════════╤═══════════════╤═══════════════╤═══════════════╤═══════════════╤══════════════╤═══════════════╤═══════════════╤═══════════════╤═══════════════╤═══════════════╤═══════════════╤═══════════════╤══════════════╤══════════════╤═══════════════╤═══════════════╤═══════════════╤═════════════╤═════════════╤═══════════════╤═════════════╤═══════════════╤══════════════╤═══════════════╤═════════════╤═════════════╤═════════════╤══════════════╤═════════════╤═════════════╤═════════════╤═════════════╤══════════════╤═════════════╤═════════════╤══════════════╤═════════════╤═════════════╤═════════════╤══════════════╤═════════════╤═══════════════╤═══════════════╤══════════════╤═════════════╤═════════════╤═══════════════╤═════════════╤═════════════╕
│ Index Code   │ Column 1      │ Column 2      │ Column 3      │ Column 4      │ Column 5     │ Column 6      │ Column 7      │ Column 8      │ Column 9      │ Column 10     │ Column 11     │ Column 12     │ Column 13    │ Column 14    │ Column 15 

############## GFS LOGIC FOR STOCKS - TESTING ###############

In [10]:
import pandas as pd
import datetime as dt
from pathlib import Path
import pandas_ta as ta
from tabulate import tabulate
import os
import yfinance as yf  # Import yfinance to fetch company names

def append_row(df, row):
    """Appends a new row to a pandas DataFrame if the row is not empty."""
    if row.isnull().all():
        return df  # Do not append if the row is all NA
    return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)

def get_company_name(symbol):
    """Fetches the company name from Yahoo Finance."""
    try:
        stock_info = yf.Ticker(symbol).info
        return stock_info.get('longName', 'N/A')  # Return 'N/A' if company name is not found
    except Exception as e:
        return 'N/A'  # Return 'N/A' in case of an error

def getRSI14_and_BB(csvfilename):
    """Calculates RSI (14 period) and Bollinger Bands (20 period, 2 std.dev) for a given CSV file."""
    if Path(csvfilename).is_file():
        try:
            df = pd.read_csv(csvfilename)
            if df.empty or 'Close' not in df.columns:
                return 0.00, 0.00, 0.00, 0.00
            else:
                df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
                df['rsi14'] = ta.rsi(df['Close'], length=14)
                bb = ta.bbands(df['Close'], length=20)
                if bb is None or df['rsi14'] is None:
                    return 0.00, 0.00, 0.00, 0.00
                df['lowerband'] = bb['BBL_20_2.0']
                df['middleband'] = bb['BBM_20_2.0']
                if pd.isna(df['rsi14'].iloc[-1]) or pd.isna(df['lowerband'].iloc[-1]) or pd.isna(df['middleband'].iloc[-1]):
                    return 0.00, 0.00, 0.00, 0.00
                else:
                    rsival = df['rsi14'].iloc[-1].round(2)
                    ltp = df['Close'].iloc[-1].round(2)
                    lowerband = df['lowerband'].iloc[-1].round(2)
                    middleband = df['middleband'].iloc[-1].round(2)
                    return rsival, ltp, lowerband, middleband
        except Exception as e:
            return 0.00, 0.00, 0.00, 0.00
    else:
        return 0.00, 0.00, 0.00, 0.00

def dayweekmonth_datasets(symbol, symbolname, index_code):
    """Calculates RSI, Bollinger Bands, and other metrics for daily, weekly, and monthly data."""
    # Replace periods with underscores in the symbol for file naming
    symbol_with_underscore = symbol.replace('.', '_')

    # Construct the file paths using the modified symbol
    daylocationstr = f'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Daily_data\\{symbol_with_underscore}.csv'
    weeklocationstr = f'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Weekly_data\\{symbol_with_underscore}.csv'
    monthlocationstr = f'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Monthly_data\\{symbol_with_underscore}.csv'

    cday = dt.datetime.today().strftime('%d/%m/%Y')
    dayrsi14, dltp, daylowerband, daymiddleband = getRSI14_and_BB(daylocationstr)
    weekrsi14, wltp, weeklowerband, weekmiddleband = getRSI14_and_BB(weeklocationstr)
    monthrsi14, mltp, monthlowerband, monthmiddleband = getRSI14_and_BB(monthlocationstr)

    company_name = get_company_name(symbol)  # Fetch company name

    new_row = pd.Series({
        'entrydate': cday,
        'indexcode': index_code,
        'indexname': symbolname,
        'companyname': company_name,  # Add company name to the new row
        'dayrsi14': dayrsi14,
        'weekrsi14': weekrsi14,
        'monthrsi14': monthrsi14,
        'dltp': dltp,
        'daylowerband': daylowerband,
        'daymiddleband': daymiddleband,
        'weeklowerband': weeklowerband,
        'weekmiddleband': weekmiddleband,
        'monthlowerband': monthlowerband,
        'monthmiddleband': monthmiddleband
    })
    return new_row

def generateGFS(scripttype):
    """Generates the GFS report based on the provided scripttype."""
    indicesdf = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'companyname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

    fname = f'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\{scripttype}.csv'
    csvfilename = f'GFS_{scripttype}.csv'
    try:
        with open(fname) as f:
            for line in f:
                if "," not in line:
                    continue
                symbol, symbolname = line.split(",")[0], line.split(",")[1]
                symbol = symbol.strip()
                new_row = dayweekmonth_datasets(symbol, symbolname, symbol)
                indicesdf = append_row(indicesdf, new_row)
    except Exception as e:
        pass

    return indicesdf

def read_indicesstocks(file_path):
    """Reads the indicesstocks.csv file and returns a dictionary."""
    indices_dict = {}
    try:
        with open(file_path, 'r') as file:
            for line in file:
                parts = line.strip().split(',')
                if len(parts) > 1:
                    index_code = parts[0].strip()
                    stocks = [stock.strip() for stock in parts[1:]]
                    indices_dict[index_code] = stocks
    except Exception as e:
        pass
    return indices_dict


# Main execution
# Ensure 'DATASETS' directory exists
os.makedirs('C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS', exist_ok=True)
os.makedirs('C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Daily_data', exist_ok=True)
os.makedirs('C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Weekly_data', exist_ok=True)
os.makedirs('C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\Monthly_data', exist_ok=True)

indicesdf_path = r'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\indicesdf.csv'
indicesstocks_path = r'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\indicesstocks.csv'
filtered_indices_path = r'C:\\Users\\manoj\\Downloads\\Major project data\\Major pro source codes\\DATASETS\\filtered_indices_output.csv'  # Corrected path

# Generate GFS report
df3 = generateGFS('indicesdf')

# Filter based on criteria
df4 = df3.loc[
    df3['monthrsi14'].between(40, 60) &
    df3['weekrsi14'].between(40, 60) &
    df3['dayrsi14'].between(40, 60)
]
# Extract only the indexcode for the filtered indices
filtered_indexcodes = df4[['indexcode']]

# Save the filtered indexcodes to a new CSV file, overwriting any existing file
filtered_indexcodes.to_csv(filtered_indices_path, index=False)

# Check if any indices qualified
if filtered_indexcodes.empty:
    print("\033[1mNO INDICES QUALIFIES THE GFS CRITERIA\033[0m")
else:
    print(tabulate(df4, headers='keys', tablefmt='fancy_grid', showindex=False))
    
    # Read indices from indicesstocks
    indicesstocks = read_indicesstocks(indicesstocks_path)

    # Read filtered indices
    filtered_indices = filtered_indexcodes['indexcode'].tolist()

    # Flag to track if any stocks qualify
    any_stocks_qualified = False

    # DataFrame to store results for qualified stocks
    results_df = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'companyname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

    # Compare and run GFS for matched indices
    for index in filtered_indices:
        if index in indicesstocks:
            stocks = indicesstocks[index]  # Get the list of stocks for the matched index 

            # Traverse through the stocks starting from the next position
            for stock in stocks[1:]:  # Start from the second stock (index + 1)
                if stock:  # Check if stock is not empty
                    matched_row = dayweekmonth_datasets(stock, stock, index)  # Using stock as both symbol and symbolname and passing index
                    results_df = append_row(results_df, matched_row)  # Append the result to the DataFrame
                    any_stocks_qualified = True

    # Save the results to a CSV file
    if not results_df.empty:
        results_df1 = results_df.loc[
            results_df['monthrsi14'].between(40, 60) &
            results_df['weekrsi14'].between(40, 60) &
            results_df['dayrsi14'].between(40, 60)
        ]
        results_df1.to_csv(filtered_indices_path, index=False)
        print(tabulate(results_df1, headers='keys', tablefmt='fancy_grid', showindex=False))
    else:
        print("\033[1mNO STOCKS QUALIFY THE GFS CRITERIA\033[0m")


C:\Users\manoj\AppData\Local\Temp\ipykernel_20992\129142517.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)


╒═════════════╤═════════════╤═════════════╤═══════════════╤════════════╤═════════════╤══════════════╤══════════╤════════════════╤═════════════════╤═════════════════╤══════════════════╤══════════════════╤═══════════════════╕
│ entrydate   │ indexcode   │ indexname   │ companyname   │   dayrsi14 │   weekrsi14 │   monthrsi14 │     dltp │   daylowerband │   daymiddleband │   weeklowerband │   weekmiddleband │   monthlowerband │   monthmiddleband │
╞═════════════╪═════════════╪═════════════╪═══════════════╪════════════╪═════════════╪══════════════╪══════════╪════════════════╪═════════════════╪═════════════════╪══════════════════╪══════════════════╪═══════════════════╡
│ 14/02/2025  │ ^NSEI       │ NIFTY 50    │ NIFTY 50      │      40.63 │       41.24 │        59.26 │ 23031.4  │       22762.5  │        23261.1  │        22762.2  │         23985.8  │         18465.5  │          22464.5  │
├─────────────┼─────────────┼─────────────┼───────────────┼────────────┼─────────────┼──────────────┼───

C:\Users\manoj\AppData\Local\Temp\ipykernel_20992\129142517.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)


╒═════════════╤═════════════╤═══════════════╤═════════════════════════════════════╤════════════╤═════════════╤══════════════╤═════════╤════════════════╤═════════════════╤═════════════════╤══════════════════╤══════════════════╤═══════════════════╕
│ entrydate   │ indexcode   │ indexname     │ companyname                         │   dayrsi14 │   weekrsi14 │   monthrsi14 │    dltp │   daylowerband │   daymiddleband │   weeklowerband │   weekmiddleband │   monthlowerband │   monthmiddleband │
╞═════════════╪═════════════╪═══════════════╪═════════════════════════════════════╪════════════╪═════════════╪══════════════╪═════════╪════════════════╪═════════════════╪═════════════════╪══════════════════╪══════════════════╪═══════════════════╡
│ 14/02/2025  │ ^NSEI       │ BAJAJ-AUTO.NS │ Bajaj Auto Limited                  │      46.22 │       41.62 │        57.46 │ 8691.35 │        8249.99 │         8704.11 │         7596.58 │          9454.59 │          4075.15 │           8156.76 │
├───────────

*****GFS LOGIC FOR STOCKS - CORRECT CODE IS DOWN BELOW*****

In [ ]:
# import pandas as pd
# import datetime as dt
# from pathlib import Path
# import pandas_ta as ta
# from tabulate import tabulate

# def append_row(df, row):
#     """Appends a new row to a pandas DataFrame if the row is not empty."""
#     if row.isnull().all():
#         return df  # Do not append if the row is all NA
#     return pd.concat([df, pd.DataFrame([row], columns=row.index)]).reset_index(drop=True)

# def getRSI14_and_BB(csvfilename):
#     """Calculates RSI (14 period) and Bollinger Bands (20 period, 2 std.dev) for a given CSV file."""
#     if Path(csvfilename).is_file():
#         try:
#             df = pd.read_csv(csvfilename)
#             if df.empty or 'Close' not in df.columns:
#                 return 0.00, 0.00, 0.00, 0.00
#             else:
#                 df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
#                 df['rsi14'] = ta.rsi(df['Close'], length=14)
#                 bb = ta.bbands(df['Close'], length=20)
#                 if bb is None or df['rsi14'] is None:
#                     return 0.00, 0.00, 0.00, 0.00
#                 df['lowerband'] = bb['BBL_20_2.0']
#                 df['middleband'] = bb['BBM_20_2.0']
#                 if pd.isna(df['rsi14'].iloc[-1]) or pd.isna(df['lowerband'].iloc[-1]) or pd.isna(df['middleband'].iloc[-1]):
#                     return 0.00, 0.00, 0.00, 0.00
#                 else:
#                     rsival = df['rsi14'].iloc[-1].round(2)
#                     ltp = df['Close'].iloc[-1].round(2)
#                     lowerband = df['lowerband'].iloc[-1].round(2)
#                     middleband = df['middleband'].iloc[-1].round(2)
#                     return rsival, ltp, lowerband, middleband
#         except Exception as e:
#             print(f"Error reading {csvfilename}: {e}")
#             return 0.00, 0.00, 0.00, 0.00
#     else:
#         print(f"File does not exist: {csvfilename}")
#         return 0.00, 0.00, 0.00, 0.00

# def dayweekmonth_datasets(symbol, symbolname):
#     """Calculates RSI, Bollinger Bands, and other metrics for daily, weekly, and monthly data."""
#     # Replace periods with underscores in the symbol for file naming
#     symbol_with_underscore = symbol.replace('.', '_')

#     # Construct the file paths using the modified symbol
#     daylocationstr = f'DATASETS/Daily_data/{symbol_with_underscore}.csv'
#     weeklocationstr = f'DATASETS/Weekly_data/{symbol_with_underscore}.csv'
#     monthlocationstr = f'DATASETS/Monthly_data/{symbol_with_underscore}.csv'

#     cday = dt.datetime.today().strftime('%d/%m/%Y')
#     dayrsi14, dltp, daylowerband, daymiddleband = getRSI14_and_BB(daylocationstr)
#     weekrsi14, wltp, weeklowerband, weekmiddleband = getRSI14_and_BB(weeklocationstr)
#     monthrsi14, mltp, monthlowerband, monthmiddleband = getRSI14_and_BB(monthlocationstr)


#   new_row = pd.Series({
#       'entrydate': cday,
#        'indexcode': index_code,
#        'indexname': symbolname,
#        'dayrsi14': dayrsi14,
#        'weekrsi14': weekrsi14,
#        'monthrsi14': monthrsi14,
#        'dltp': dltp,
#        'daylowerband': daylowerband,
#        'daymiddleband': daymiddleband,
#        'weeklowerband': weeklowerband,
#        'weekmiddleband': weekmiddleband,
#        'monthlowerband': monthlowerband,
#        'monthmiddleband': monthmiddleband
#    })
#    return new_row

# def generateGFS(scripttype):
#     """Generates the GFS report based on the provided scripttype."""
#     indicesdf = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

#     fname = f'DATASETS/{scripttype}.csv'
#     try:
#         with open(fname) as f:
#             for line in f:
#                 if "," not in line:
#                     continue
#                 symbol, symbolname = line.split(",")[0], line.split(",")[1]
#                 symbol = symbol.strip()
#                 new_row = dayweekmonth_datasets(symbol, symbolname)
#                 indicesdf = append_row(indicesdf, new_row)
#     except Exception as e:
#         print(f"Error processing {fname}: {e}")

#     return indicesdf

# # Main execution
# indicesdf_path = r'DATASETS/indicesdf.csv'
# indicesstocks_path = r'C:\Users\manoj\Downloads\Major project data\Major pro source codes\DATASETS\indicesstocks.csv'
# filtered_indices_path = r'C:\Users\manoj\Downloads\Major project data\Major pro source codes\DATASETS\filtered_indices_output.csv'  # Corrected path

# # Generate GFS report
# df3 = generateGFS('indicesdf')

# # Filter based on criteria
# df4 = df3.loc[
#     (df3['monthrsi14'] >= 60 ) &
#     (df3['weekrsi14']  >= 60 ) &
#     df3['dayrsi14'].between(40, 50) &
#     (df3['dltp'] > df3['daylowerband']) &
#     (df3['dltp'] < df3['daymiddleband'])
# ]
# # Extract only the indexcode for the filtered indices
# filtered_indexcodes = df4[['indexcode']]

# # Save the filtered indexcodes to a new CSV file, overwriting any existing file
# filtered_indexcodes.to_csv(filtered_indices_path, index=False)

# # Check if any indices qualified
# if filtered_indexcodes.empty:
#     print("\033[1mNO STOCKS QUALIFY THE GFS CRITERIA\033[0m")
# else:
#     # Read indices from indicesstocks
#     indicesstocks = read_indicesstocks(indicesstocks_path)

#     # Read filtered indices
#     filtered_indices = filtered_indexcodes['indexcode'].tolist()

#     # Flag to track if any stocks qualify
#     any_stocks_qualified = False

#     # DataFrame to store results for qualified stocks
#     results_df = pd.DataFrame(columns=['entrydate', 'indexcode', 'indexname', 'dayrsi14', 'weekrsi14', 'monthrsi14', 'dltp', 'daylowerband', 'daymiddleband', 'weeklowerband', 'weekmiddleband', 'monthlowerband', 'monthmiddleband'])

#     # Compare and run GFS for matched indices
#     for index in filtered_indices:
#         if index in indicesstocks:
#             stocks = indicesstocks[index]  # Get the list of stocks for the matched index 

#             # Traverse through the stocks starting from the next pAosition
#             for stock in stocks[1:]:  # Start from the second stock (index + 1)
#                 if stock:  # Check if stock is not empty
#                     matched_row = dayweekmonth_datasets(stock, stock)  # Using stock as both symbol and symbolname
#                     results_df = append_row(results_df, matched_row)  # Append the result to the DataFrame
#                     any_stocks_qualified = True

#     # Save the results to a CSV file
#     if not results_df.empty:
#         results_df.to_csv(filtered_indices_path, index=False)
#         print("Results saved to:", filtered_indices_path)
#         # Print the results to the console
#         print(tabulate(results_df, headers='keys', tablefmt='fancy_grid', showindex=False))
#     else:
#        print("\033[1mNO STOCKS QUALIFY THE GFS CRITERIA\033[0m")